# Configuración e importación de un modelo de robot desde Onshape hacia MuJoCo

Este notebook establece el flujo de trabajo completo para:
1. Instalar las librerías necesarias.
2. Configurar las credenciales de la API de Onshape.
3. Definir un archivo de configuración (`config.json`) avanzado que especifica:
   - URL del ensamblaje en Onshape.
   - Propiedades dinámicas de las articulaciones (actuación, límites, ganancias PID, fricción, etc.).
   - Formato de salida (MuJoCo).
   - Opciones de mallas visuales y de colisión.
4. Generar automáticamente el modelo MJCF mediante `onshape-to-robot`.
5. Visualizar y simular el robot en un entorno interactivo de MuJoCo, con control de parámetros.

El procedimiento sigue prácticas de ingeniería de software: verificación de dependencias, manejo de excepciones y cierre adecuado de recursos.

## 1. Instalación de la librería `onshape-to-robot`

Esta herramienta permite extraer ensamblajes desde Onshape y convertirlos a formatos compatibles con motores de simulación.  
Documentación oficial: [onshape-to-robot GitHub](https://github.com/RobotecAI/onshape-to-robot)

Se instala silenciosamente y se verifica la versión.

In [ ]:
import sys
import subprocess

# Instalar onshape-to-robot
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onshape-to-robot", "-q"])
    print("Onshape-to-robot instalado correctamente.")
except subprocess.CalledProcessError as e:
    print(f"Error durante la instalación: {e}")
    sys.exit(1)

from importlib.metadata import version
print(f"Versión de onshape-to-robot: {version('onshape-to-robot')}")

## 2. Configuración de claves de acceso a la API de Onshape

Onshape requiere autenticación mediante `access_key` y `secret_key`.  
Se obtienen desde el [Portal de Desarrollador de Onshape](https://dev-portal.onshape.com/keys).  
Se configuran como variables de entorno.

In [ ]:
import os
from pathlib import Path

# Cargar variables de entorno desde archivo .env (opcional)
env_file = Path(".env")
if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file)
else:
    print("⚠️  Archivo .env no encontrado. Asegúrate de configurar las variables de entorno.")
    print("   Copia .env.example a .env y actualiza con tus credenciales reales.")

# Obtener credenciales desde variables de entorno
ONSHAPE_API = os.getenv('ONSHAPE_API', 'https://cad.onshape.com')
ONSHAPE_ACCESS_KEY = os.getenv('ONSHAPE_ACCESS_KEY')
ONSHAPE_SECRET_KEY = os.getenv('ONSHAPE_SECRET_KEY')

# Validar que las credenciales estén configuradas
if not ONSHAPE_ACCESS_KEY or not ONSHAPE_SECRET_KEY:
    raise RuntimeError(
        "❌ Credenciales de Onshape no configuradas.\n"
        "   1. Copia .env.example a .env\n"
        "   2. Obtén tus claves desde: https://dev-portal.onshape.com/keys\n"
        "   3. Actualiza ONSHAPE_ACCESS_KEY y ONSHAPE_SECRET_KEY en .env"
    )

# Configurar variables de entorno para la herramienta
os.environ['ONSHAPE_API'] = ONSHAPE_API
os.environ['ONSHAPE_ACCESS_KEY'] = ONSHAPE_ACCESS_KEY
os.environ['ONSHAPE_SECRET_KEY'] = ONSHAPE_SECRET_KEY


print("✅ Credenciales de Onshape configuradas desde variables de entorno.")

## 3. Configuración avanzada del proyecto: `model/config.json`

El archivo de configuración permite controlar aspectos finos de la generación del modelo y la simulación.  
**Parámetros incluidos:**

- `url`: URL del ensamblaje en Onshape.
- `output_format`: Formato de salida (`mujoco`).
- `joint_propieties`: (Nótese que la clave correcta según la documentación es `joint_properties`; se usará la correcta internamente).  
  Para todas las articulaciones (`"*"`) se definen:
  - `actuated`: si el joint tiene actuador.
  - `forcerange`: par máximo (N·m).
  - `limits`: [mínimo, máximo] en radianes.
  - `frictionloss`: fricción viscosa.
  - `kv`: ganancia de velocidad.
  - `kp`: ganancia proporcional (posición).
- `output_filename`: nombre base del archivo de salida (sin extensión).
- `use_visual_meshes`: usar mallas visuales.
- `use_collision_meshes`: usar mallas de colisión.
- `simplify_collision_meshes`: simplificar geometrías de colisión.

> **Nota**: La clave `joint_propieties` se ha renombrado a `joint_properties` en el código para cumplir con el estándar de `onshape-to-robot`.

In [ ]:
import json
import os
import sys

# URL del ensamblaje proporcionada
ASSEMBLY_URL = "https://cad.onshape.com/documents/d578f2fbdc332270e30f24e5/w/ba84a5b38e318fd89e8c140d/e/9356b596fb5246da0639ba81"

# Configuración completa según especificación, usando la clave correcta "joint_properties"
config = {
    "url": ASSEMBLY_URL,
    "output_format": "mujoco",
    "output_filename": "HexaBot",
    "use_visual_meshes": True,
    "use_collision_meshes": True,
    "simplify_collision_meshes": True
}

# Crear directorio 'model'
os.makedirs("model", exist_ok=True)

config_path = os.path.join("model", "config.json")
try:
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    print(f"Archivo de configuración enriquecido creado en: {config_path}")
    print("   Parámetros configurados:")
    for key, value in config.items():
        print(f"     - {key}: {value}")
except IOError as e:
    print(f"Error al escribir el archivo: {e}")
    sys.exit(1)

## 4. Importación del robot desde Onshape

Se ejecuta `onshape-to-robot` sobre el directorio `model`.  
La herramienta leerá `config.json` y generará los archivos:
- `HexaBot.xml` (modelo principal)
- `scene.xml` (escena con suelo y luces)
- Mallas STL en `model/assets/`

Si alguna articulación tiene propiedades específicas, serán aplicadas en el archivo XML.

In [ ]:
import subprocess
import sys

try:
    result = subprocess.run(
        ["onshape-to-robot", "model"],
        capture_output=True,
        text=True,
        check=False
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("Falló la generación del modelo:")
        print(result.stderr)
        sys.exit(result.returncode)
    else:
        print("Modelo generado exitosamente con la configuración enriquecida.")
except FileNotFoundError:
    print("El comando 'onshape-to-robot' no se encuentra.")
    sys.exit(1)
except Exception as e:
    print(f"Error inesperado: {e}")
    sys.exit(1)

## 5. Carga y simulación interactiva en MuJoCo

Se carga el modelo generado (`HexaBot.xml` o `scene.xml`).  
Se inicializan los actuadores con los valores por defecto (las ganancias `kp`, `kv`, `forcerange` ya están en el XML).  
Se implementa un visor robusto que maneja el cierre correcto incluso en versiones antiguas de MuJoCo.

In [ ]:
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mujoco", "-q"])
    print("MuJoCo instalado correctamente.")
except subprocess.CalledProcessError as e:
    print(f"Error instalando MuJoCo: {e}")
    sys.exit(1)

from importlib.metadata import version
print(f"Versión de MuJoCo: {version('mujoco')}")

In [ ]:
import mujoco
import mujoco.viewer
import signal
import sys

MODEL_XML = "model/scene.xml"  # Se usa este porque el HexaBot.xml solo contiene el robot,
                                # y el scene.xml es el que se genera con el suelo.

try:
    model = mujoco.MjModel.from_xml_path(MODEL_XML) #type: ignore
    data = mujoco.MjData(model) #type: ignore
    print(f"Modelo cargado desde {MODEL_XML}")
    print(f"   Número de articulaciones: {model.njnt}")
    print(f"   Número de actuadores: {model.nu}")
except Exception as e:
    print(f"No se pudo cargar el modelo: {e}")
    sys.exit(1)

# Inicializar controles
for i in range(model.nu):
    data.ctrl[i] = 0.0

viewer = None

def signal_handler(sig, frame):
    print("\nCerrando visor...")
    if viewer is not None:
        viewer.close()
    sys.exit(0)

signal.signal(signal.SIGINT, signal_handler)

try:
    # Intento con gestor de contexto
    with mujoco.viewer.launch(model, data) as viewer: #type: ignore
        print("Visor interactivo abierto. Presiona Ctrl+C o cierra la ventana.")
        while viewer.is_running():
            mujoco.mj_step(model, data) #type: ignore
            viewer.sync()
except AttributeError:
    # Fallback a launch_passive
    print("Usando modo pasivo (launch_passive).")
    viewer = mujoco.viewer.launch_passive(model, data)
    while viewer.is_running():
        mujoco.mj_step(model, data) #type: ignore
        viewer.sync()
    viewer.close()
except Exception as e:
    print(f"Error: {e}")